In [0]:
dbutils.library.restartPython()

# Spike — validação de chaves únicas para MERGE

Confirma que as 3 tabelas identificadas sem chave verdadeiramente única (`erp_posicoes_estoque`, `crm_itens_pedido`, `tms_leituras_temperatura`) passam a ter, após a correção que adiciona `posicao_id`, `item_pedido_id` e `leitura_id` respectivamente — pré-requisito para `MERGE INTO` funcionar sem erro de múltiplas linhas casando com a mesma chave.

In [0]:
%pip install dbldatagen Faker

In [0]:
from datetime import date
from src.simuladores.simulador_erp import SimuladorERP

simulador = SimuladorERP(spark=spark, dbutils=dbutils)
resultado = simulador.gerar_dia(date(2026, 8, 5))
print(resultado)

In [0]:
caminho = simulador.caminho_landing(date(2026, 8, 5))
conteudo = dbutils.fs.head(f"{caminho}/erp_posicoes_estoque.json", 800)
print(conteudo)

In [0]:
from datetime import date
from src.simuladores.simulador_crm import SimuladorCRM

simulador_crm = SimuladorCRM(spark=spark, dbutils=dbutils)
resultado = simulador_crm.gerar_dia(date(2026, 8, 5))
print(resultado)

In [0]:
caminho_crm = simulador_crm.caminho_landing(date(2026, 8, 5))
conteudo = dbutils.fs.head(f"{caminho_crm}/crm_itens_pedido.json", 800)
print(conteudo)

In [0]:
from datetime import date
from src.simuladores.simulador_tms import SimuladorTMS

simulador_tms = SimuladorTMS(spark=spark, dbutils=dbutils)
resultado = simulador_tms.gerar_dia(date(2026, 8, 3))
print(resultado)

In [0]:
caminho_tms = simulador_tms.caminho_landing(date(2026, 8, 3))
conteudo = dbutils.fs.head(f"{caminho_tms}/tms_leituras_temperatura.json", 800)
print(conteudo)

In [0]:
from datetime import date
from src.transformacao.configuracao_tabelas import CONFIGURACAO_TABELAS
from src.transformacao.transformar_bronze_para_silver import transformar_bronze_para_silver

resultados = []
for tabela, config in CONFIGURACAO_TABELAS.items():
    resultado = transformar_bronze_para_silver(
        spark=spark, catalog="poc_pulse_observability", tabela=tabela, config=config
    )
    resultados.append(resultado)
    print(resultado)

In [0]:
resultados = []
for tabela, config in CONFIGURACAO_TABELAS.items():
    resultado = transformar_bronze_para_silver(
        spark=spark, catalog="poc_pulse_observability", tabela=tabela, config=config
    )
    resultados.append(resultado)
    print(resultado)

In [0]:
tabelas_afetadas = ["erp_posicoes_estoque", "crm_itens_pedido", "tms_leituras_temperatura"]

for tabela in tabelas_afetadas:
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.bronze.{tabela}")
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.silver.{tabela}")
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/_autoloader_checkpoint/{tabela}", recurse=True)
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/_autoloader_schema/{tabela}", recurse=True)

print("Reset concluído para:", tabelas_afetadas)

In [0]:
datas_ja_geradas = ["2026-08-03", "2026-08-07", "2026-08-08", "2026-08-10"]

for data_str in datas_ja_geradas:
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/erp/data={data_str}/erp_posicoes_estoque.json")
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/crm/data={data_str}/crm_itens_pedido.json")
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/tms/data={data_str}/tms_leituras_temperatura.json")

print("Arquivos antigos removidos.")

In [0]:
todas_as_tabelas = [
    "erp_lotes_producao", "erp_posicoes_estoque", "erp_notas_expedicao",
    "crm_pedidos", "crm_itens_pedido", "crm_atendimento",
    "tms_remessas", "tms_leituras_temperatura", "tms_comprovantes_entrega",
    "financeiro_faturas", "financeiro_contas_receber",
]

for tabela in todas_as_tabelas:
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.bronze.{tabela}")
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.silver.{tabela}")
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/_autoloader_checkpoint/{tabela}", recurse=True)
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/_autoloader_schema/{tabela}", recurse=True)

print("Reset completo concluído.")

In [0]:
datas_ja_geradas = ["2026-08-03", "2026-08-07", "2026-08-08", "2026-08-10"]
sistemas = ["erp", "crm", "tms", "financeiro"]

for sistema in sistemas:
    for data_str in datas_ja_geradas:
        caminho = f"/Volumes/poc_pulse_observability/landing/raw/{sistema}/data={data_str}"
        dbutils.fs.rm(caminho, recurse=True)

print("Landing Zone limpa para as 4 datas.")

In [0]:
from src.simuladores.simulador_factory import SimuladorFactory

for data_str in ["2026-08-03", "2026-08-07", "2026-08-08", "2026-08-10"]:
    data_ref = date.fromisoformat(data_str)
    print(f"--- Gerando {data_str} ---")
    for nome_sistema in SimuladorFactory.ordem_execucao():
        simulador = SimuladorFactory.criar(nome_sistema, spark=spark, dbutils=dbutils)
        simulador.executar_seed()
        resultado = simulador.gerar_dia(data_ref)
        print(f"  {resultado['sistema']}: {resultado['status']} — {resultado.get('tabelas_geradas', [])}")

In [0]:
from src.ingestao.ingestor_autoloader import IngestorAutoloader

TABELAS_POR_SISTEMA = {
    "erp": ["erp_lotes_producao", "erp_posicoes_estoque", "erp_notas_expedicao"],
    "crm": ["crm_pedidos", "crm_itens_pedido", "crm_atendimento"],
    "tms": ["tms_remessas", "tms_leituras_temperatura", "tms_comprovantes_entrega"],
    "financeiro": ["financeiro_faturas", "financeiro_contas_receber"],
}

for sistema, tabelas in TABELAS_POR_SISTEMA.items():
    for tabela in tabelas:
        ingestor = IngestorAutoloader(spark=spark, sistema=sistema, tabela=tabela)
        resultado = ingestor.executar()
        print(resultado)

In [0]:
from src.transformacao.configuracao_tabelas import CONFIGURACAO_TABELAS
from src.transformacao.transformar_bronze_para_silver import transformar_bronze_para_silver

for tabela, config in CONFIGURACAO_TABELAS.items():
    resultado = transformar_bronze_para_silver(
        spark=spark, catalog="poc_pulse_observability", tabela=tabela, config=config
    )
    print(resultado)

In [0]:
for tabela, config in CONFIGURACAO_TABELAS.items():
    resultado = transformar_bronze_para_silver(
        spark=spark, catalog="poc_pulse_observability", tabela=tabela, config=config
    )
    print(resultado)